# R1 — (G, in-strength) phase diagram

Companion analysis to Fig. 5 of the manuscript, requested by Reviewer B (point 3): a 2D phase diagram of AC1$(R_n)$ in the (G, in-strength) parameter space, plus three quantitative summaries of that plane.

**Strategy.** No new simulations. Per-region AC1 is computed from the existing whole-brain G-sweep (16 G values × 3 regimes). The figure is then a re-aggregation of these data.

**Inputs.**
- Whole-brain simulation outputs at `$MBC_SIM_ROOT/Gscan_connectome_{sub,crit,super}/data_G%d.pkl`.

**Outputs.**
- `revision/derived_data/AC1_local.npy` — shape `(3, 60, 16)`.
- `revision/derived_data/{rho_instr_AC1,non_monotonicity_score_crit,var_AC1_per_G}.npy`.
- `revision/figures/Fig_B3_phase_diagram.{pdf,png}` — main supplementary figure.
- `revision/figures/Fig_B3_SI_VISal_{PSD,avalanches}.{pdf,png}` — high-in-strength contrast exemplar (companion supplementary panels).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# run from revision/round1/
sys.path.insert(0, str(Path().resolve()))          # revision_utils, run_*.py
sys.path.insert(0, str(Path().resolve().parent))   # revision/paths.py
import paths
sys.path.insert(0, str(paths.SRC))                 # original src/

from revision_utils import (
    G_GRID,
    G_WORKING_INDEX,
    HIGHEST_INSTRENGTH_IDX,
    HIGHEST_INSTRENGTH_NAME,
    INSTRENGTH,
    REGIMES,
    compute_AC1_local,
)

DERIVED = paths.DERIVED
FIG_DIR = paths.FIGURES
DERIVED.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

AC1_PATH = DERIVED / 'AC1_local.npy'

## 1. Compute or load AC1 cache

AC1 with lag 20 (matches the published Fig. 5A definition: `functions.AC1(R, n=20)` from `src/functions.py`).

In [ ]:
if AC1_PATH.exists():
    AC1_local = np.load(AC1_PATH)
    print(f'Loaded cached AC1_local from {AC1_PATH}, shape={AC1_local.shape}')
else:
    AC1_local = compute_AC1_local()
    np.save(AC1_PATH, AC1_local)
    print(f'Saved {AC1_PATH}')

## 2. Sanity checks

At G=0 all regions are identical and uncoupled, so AC1(R_n) should be tightly clustered.

In [ ]:
for ir, name in enumerate(REGIMES):
    print(f'{name:13s}  G=0   mean={AC1_local[ir,:,0].mean():+.3f}  std={AC1_local[ir,:,0].std():.3f}')
    gw = G_WORKING_INDEX[name]
    print(f'{name:13s}  G*[{gw:2d}] mean={AC1_local[ir,:,gw].mean():+.3f}  std={AC1_local[ir,:,gw].std():.3f}')

## 3. Build the main supplementary figure

Layout (2 rows × 3 cols):

- **Row 1** — per-regime binned heatmap of $\langle$AC1$(R_n)\rangle$ over (G, in-strength bin). 8 quantile bins of in-strength.
- **Row 2** — three quantitative summaries of the (G, in-strength) plane:
    - (i) Spearman $\rho$ between in-strength and AC1, as a function of G. ρ is computed directly from the per-region AC1 values (one ρ per regime per G); the binned heatmap and ρ are two views of the same data.
    - (ii) Per-region non-monotonicity score $\max_G\, \mathrm{AC1}(G) - \mathrm{AC1}(G=0)$ in the **critical** regime, plotted vs. in-strength.
    - (iii) Cross-region variance $\mathrm{Var}_n[\mathrm{AC1}(R_n)]$ vs. G, one curve per regime.

Working points $G^\ast$ are marked as dashed vertical lines.

In [ ]:
import importlib, run_B3_make_figure as fig_mod
importlib.reload(fig_mod)
fig_mod.main()

In [ ]:
from IPython.display import Image
Image(str(FIG_DIR / 'Fig_B3_phase_diagram.png'))

## 4. Companion supplementary panels: high-in-strength contrast exemplar

Fig. 5C in the main manuscript uses retrosplenial-v as the exemplar (DMN node, biological motivation; in-strength ≈ 2.03 — upper-middle rank).

To verify that the in-strength gradient identified in the (G, in-strength) phase diagram (panels (i)–(iii) above) materialises in time-series statistics, we produce the same PSD and avalanche P(S) panels at the cortical region of **highest** in-strength: the **Right Anterolateral visual area** (`VISal`, cortical idx 10, in-strength = 4.41 — the maximum).

Style is identical to Fig. 5C in `notebook 6` (same colormaps, same G values, same Welch settings, same avalanche-extraction call). Panels are saved as separate PDFs/PNGs for direct insertion into the supplementary materials.

In [ ]:
import run_B3_VISal_panels as visal_mod
importlib.reload(visal_mod)
visal_mod.main()

In [ ]:
from IPython.display import display
display(Image(str(FIG_DIR / 'Fig_B3_SI_VISal_PSD.png')))
display(Image(str(FIG_DIR / 'Fig_B3_SI_VISal_avalanches.png')))

## 5. Notes for the response to reviewers

**Findings from the (G, in-strength) phase diagram:**
1. **In-strength gradient (panel (i)).** ρ(G) is monotone-negative in subcritical and supercritical regimes, but **non-monotonic and sign-flipping** in the critical regime. The first sign flip occurs at G\* = 0.028 — the working point that best fits BOLD FC and at which global criticality emerges.
2. **Non-monotonicity score (panel (ii)).** In the critical regime, the non-monotonicity score correlates strongly with in-strength (Pearson r = +0.81; Spearman ρ = +0.82). High-in-strength regions exhibit the strongest excursions toward criticality at intermediate coupling.
3. **Cross-region heterogeneity (panel (iii)).** In the critical regime, Var_n[AC1] is *minimal* at G\*, indicating that global coordination at G\* homogenizes regional dynamics. In sub/supercritical regimes, variance peaks well after G\*.

**Exemplar treatment.** Fig. 5C is unchanged — retrosplenial-v retained as the main-text exemplar (DMN node, biological motivation). VISal is added as a high-in-strength contrast in the supplementary; it shows the same regime-specific signatures but more pronounced (sharper oscillatory peaks in supercritical, more peaked avalanche distributions), confirming the in-strength gradient predicted by the phase diagram.

**Caption fix (Fig. 5B).** The published caption refers to ΔAC1, but the panel actually plots raw AC1 at G\*. The caption is corrected accordingly (no change to the plot itself).